# 🐍 Python od podstaw — Moduł 5: Obsługa błędów i pliki

### Gdy program spotyka rzeczywistość

Dane od użytkownika, plik, którego nie ma, dzielenie przez zero — świat rzadko jest
tak czysty, jak przykłady w poprzednich modułach. Ten notebook to jak sprawić, żeby
program reagował na problemy w kontrolowany sposób, zamiast się wywalać, oraz jak
trwale zapisywać i wczytywać dane z dysku.

## Spis treści

1. [Anatomia błędu — traceback](#sec1)
2. [`try` / `except` — podstawy](#sec2)
3. [Łapanie konkretnych wyjątków](#sec3)
4. [`else` i `finally`](#sec4)
5. [Zgłaszanie własnych błędów — `raise`](#sec5)
6. [Odczyt plików](#sec6)
7. [Zapis do plików](#sec7)
8. [Ciekawostka: EAFP kontra LBYL](#sec8)
9. [Podsumowanie modułu](#sec9)
10. [Ćwiczenia](#sec10)

---

<a id="sec1"></a>
## 1. Anatomia błędu — traceback

Gdy w Pythonie coś pójdzie nie tak, program przerywa działanie i wypisuje **traceback**
— informację, w której linijce i z jakiego powodu wystąpił błąd. To nie jest coś, czego
trzeba się bać — to najważniejsza wskazówka do naprawienia kodu. Czytaj traceback **od
dołu do góry**: ostatnia linijka mówi, jaki to typ błędu i o co dokładnie chodzi.

In [ ]:
# Uruchom to celowo, żeby zobaczyć traceback:
liczba = int("nie liczba")

> 💡 **Jak czytać traceback**
>
> Ostatnia linia (`ValueError: invalid literal for int() with base 10: 'nie liczba'`) mówi wprost, co poszło nie tak. Linie wyżej pokazują ścieżkę wywołań, która do tego doprowadziła - w prostych skryptach zwykle wystarczy spojrzeć na ostatnią linię i tę wskazującą numer linii w Twoim kodzie.

<a id="sec2"></a>
## 2. `try` / `except` — podstawy

Blok `try` "próbuje" wykonać kod, a jeśli wewnątrz wystąpi błąd (wyjątek), sterowanie
przechodzi do bloku `except` zamiast przerywać cały program.

In [ ]:
try:
    liczba = int("nie liczba")
    print("Ta linia się nie wykona")
except ValueError:
    print("Nie udało się przekonwertować tekstu na liczbę")

print("Program działa dalej normalnie")

<a id="sec3"></a>
## 3. Łapanie konkretnych wyjątków

Warto łapać **konkretny typ** wyjątku, a nie wszystkie na raz (`except:` bez typu) —
dzięki temu nie ukrywasz przypadkiem zupełnie innych, niespodziewanych błędów. Można
obsłużyć kilka typów w jednym bloku, albo osobnymi blokami `except`.

In [ ]:
def podziel(a, b):
    try:
        wynik = a / b
        return wynik
    except ZeroDivisionError:
        print("Nie można dzielić przez zero!")
        return None
    except TypeError:
        print("Oba argumenty muszą być liczbami!")
        return None

print(podziel(10, 2))
print(podziel(10, 0))
print(podziel(10, "abc"))

# Kilka typów w jednym except:
try:
    slownik = {"a": 1}
    print(slownik["b"])
except (KeyError, IndexError) as blad:
    print(f"Coś nie zostało znalezione: {blad}")

> ⚠️ **Unikaj gołego `except:`**
>
> `except:` bez podania typu złapie **dosłownie każdy** błąd - łącznie z literówką w nazwie zmiennej czy próbą przerwania programu klawiszem Ctrl+C. To utrudnia wykrycie prawdziwych błędów w kodzie, bo wszystko jest po cichu połykane. Zawsze podawaj konkretny typ wyjątku, jaki się spodziewasz.

<a id="sec4"></a>
## 4. `else` i `finally`

Pełna konstrukcja `try` ma jeszcze dwa opcjonalne bloki:

- **`else`** — wykonuje się, tylko jeśli `try` zakończyło się **bez** błędu,
- **`finally`** — wykonuje się **zawsze**, niezależnie od tego, czy był błąd, czy nie
  (przydatne np. do sprzątania zasobów).

In [ ]:
def podziel(a, b):
    try:
        wynik = a / b
    except ZeroDivisionError:
        print("Błąd: dzielenie przez zero")
    else:
        print(f"Wynik: {wynik}")   # tylko gdy się udało
    finally:
        print("Koniec próby dzielenia")   # zawsze

podziel(10, 2)
print("---")
podziel(10, 0)

<a id="sec5"></a>
## 5. Zgłaszanie własnych błędów — `raise`

Czasem to Twój kod powinien zgłosić błąd, gdy dane nie spełniają jakiejś reguły
biznesowej (np. wiek nie może być ujemny). Robi się to instrukcją `raise` z odpowiednim
typem wyjątku i komunikatem.

In [ ]:
def ustaw_wiek(wiek):
    if wiek < 0:
        raise ValueError("Wiek nie może być ujemny")
    return wiek

print(ustaw_wiek(30))

try:
    ustaw_wiek(-5)
except ValueError as blad:
    print(f"Błąd walidacji: {blad}")

<a id="sec6"></a>
## 6. Odczyt plików

Plik otwiera się funkcją `open()`, najlepiej w bloku `with` — dzięki temu plik zostanie
**automatycznie zamknięty**, nawet jeśli w środku wystąpi błąd. Bez `with` trzeba by
pamiętać o ręcznym `.close()`.

In [ ]:
# Najpierw stwórzmy plik do odczytu (o zapisie - w następnej sekcji):
with open("przyklad.txt", "w", encoding="utf-8") as plik:
    plik.write("Pierwsza linia\n")
    plik.write("Druga linia\n")
    plik.write("Trzecia linia\n")

# Odczyt całości naraz:
with open("przyklad.txt", "r", encoding="utf-8") as plik:
    tresc = plik.read()
print(tresc)

# Odczyt linia po linii (przydatne dla dużych plików):
with open("przyklad.txt", "r", encoding="utf-8") as plik:
    for linia in plik:
        print("Linia:", linia.strip())   # .strip() usuwa znak nowej linii na końcu

> ⚠️ **Plik, którego nie ma**
>
> Próba otwarcia w trybie `"r"` (odczyt) pliku, który nie istnieje, wywoła `FileNotFoundError`. To dobry kandydat do obsłużenia przez `try`/`except`, szczególnie gdy nazwa pliku pochodzi od użytkownika.

<a id="sec7"></a>
## 7. Zapis do plików

Tryb otwarcia pliku decyduje, co się stanie z jego zawartością:

| Tryb | Znaczenie |
|---|---|
| `"w"` | zapis — **nadpisuje** cały plik (albo tworzy nowy) |
| `"a"` | append — dopisuje na końcu, nie kasując istniejącej treści |
| `"r"` | odczyt (domyślny, jeśli nie podasz trybu) |

In [ ]:
# "w" nadpisuje cały plik:
with open("log.txt", "w", encoding="utf-8") as plik:
    plik.write("Start programu\n")

# "a" dopisuje, nie kasując wcześniejszej treści:
with open("log.txt", "a", encoding="utf-8") as plik:
    plik.write("Zdarzenie 1\n")

with open("log.txt", "a", encoding="utf-8") as plik:
    plik.write("Zdarzenie 2\n")

with open("log.txt", "r", encoding="utf-8") as plik:
    print(plik.read())

<a id="sec8"></a>
## 8. Ciekawostka: EAFP kontra LBYL

Dwie filozofie obsługi potencjalnych błędów.

In [ ]:
slownik = {"a": 1}

# LBYL - "Look Before You Leap" (sprawdź, zanim spróbujesz):
if "b" in slownik:
    print(slownik["b"])
else:
    print("Brak klucza 'b'")

# EAFP - "Easier to Ask Forgiveness than Permission" (spróbuj, obsłuż błąd, jeśli wystąpi):
try:
    print(slownik["b"])
except KeyError:
    print("Brak klucza 'b'")

> 💡 **Ciekawostka**
>
> Python jest tradycyjnie językiem preferującym styl EAFP - `try`/`except` jest często szybsze i uważane za bardziej «pythonowe» niż seria `if`-ów sprawdzających warunki z góry, szczególnie gdy błąd jest rzadkim przypadkiem, a nie normalnym przebiegiem programu.

<a id="sec9"></a>
## 9. Podsumowanie modułu

Po tym module powinno być jasne:

- jak czytać traceback, żeby zrozumieć, co poszło nie tak,
- jak działa `try`/`except`, i dlaczego warto łapać konkretne typy wyjątków,
- do czego służą `else` i `finally` w bloku `try`,
- jak zgłaszać własne błędy przez `raise`,
- jak bezpiecznie czytać i zapisywać pliki przez `with open(...)`,
- różnicę między trybami `"r"`, `"w"` i `"a"`.

To domyka podstawowy kurs Pythona zaczęty w module 1. Kolejny naturalny krok to już
bardziej zaawansowane tematy: programowanie obiektowe (klasy), praca z modułami i
pakietami, albo konkretne biblioteki (np. do analizy danych) — w zależności, w którą
stronę chcesz pójść dalej.

<a id="sec10"></a>
## 10. Ćwiczenia

Ostatni komplet ćwiczeń w tej serii — kilka z nich łączy obsługę błędów z plikami,
żeby przećwiczyć oba tematy naraz.

> 📝 **Ćwiczenie 1: Bezpieczne dzielenie**
>
> Napisz funkcję `bezpieczne_dzielenie(a, b)`, która zwraca wynik `a / b`, a jeśli `b` wynosi 0 - łapie `ZeroDivisionError`, wypisuje komunikat i zwraca `None`.

**👉 Przykładowe rozwiązanie** *(najpierw spróbuj samodzielnie!)*

```python
def bezpieczne_dzielenie(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        print("Nie można dzielić przez zero")
        return None

print(bezpieczne_dzielenie(10, 2))
print(bezpieczne_dzielenie(10, 0))
```

> 📝 **Ćwiczenie 2: Walidacja wieku w pętli**
>
> Napisz pętlę `while True`, która pyta użytkownika o wiek przez `input()`, próbuje przekonwertować na `int`, i jeśli się nie uda (`ValueError`) - wypisuje komunikat i pyta ponownie. Kończy się, gdy użytkownik poda poprawną liczbę.

**👉 Przykładowe rozwiązanie** *(najpierw spróbuj samodzielnie!)*

```python
while True:
    tekst = input("Podaj swój wiek: ")
    try:
        wiek = int(tekst)
        print(f"Dziękuję, masz {wiek} lat")
        break
    except ValueError:
        print("To nie jest poprawna liczba, spróbuj ponownie")
```

> 📝 **Ćwiczenie 3: Kalkulator z obsługą kilku błędów**
>
> Napisz funkcję `kalkulator(a, b, operacja)`, gdzie `operacja` to string `"+"`, `"-"`, `"*"` albo `"/"`. Obsłuż `ZeroDivisionError` przy dzieleniu przez zero oraz sytuację, gdy `operacja` nie jest żadną ze znanych (zwróć wtedy komunikat o nieznanej operacji zamiast błędu).

**👉 Przykładowe rozwiązanie** *(najpierw spróbuj samodzielnie!)*

```python
def kalkulator(a, b, operacja):
    if operacja == "+":
        return a + b
    elif operacja == "-":
        return a - b
    elif operacja == "*":
        return a * b
    elif operacja == "/":
        try:
            return a / b
        except ZeroDivisionError:
            return "Błąd: dzielenie przez zero"
    else:
        return f"Nieznana operacja: {operacja}"

print(kalkulator(10, 2, "/"))
print(kalkulator(10, 0, "/"))
print(kalkulator(10, 2, "%"))
```

> 📝 **Ćwiczenie 4: Zapis listy do pliku**
>
> Mając listę `imiona = ["Kamil", "Ania", "Tomek"]`, zapisz każde imię w osobnej linii do pliku `imiona.txt`.

**👉 Przykładowe rozwiązanie** *(najpierw spróbuj samodzielnie!)*

```python
imiona = ["Kamil", "Ania", "Tomek"]

with open("imiona.txt", "w", encoding="utf-8") as plik:
    for imie in imiona:
        plik.write(imie + "\n")

print("Zapisano plik imiona.txt")
```

> 📝 **Ćwiczenie 5: Odczyt i zliczanie linii**
>
> Wczytaj plik `imiona.txt` z poprzedniego ćwiczenia i wypisz, ile linii (imion) zawiera, oraz każde imię wielkimi literami.

**👉 Przykładowe rozwiązanie** *(najpierw spróbuj samodzielnie!)*

```python
with open("imiona.txt", "r", encoding="utf-8") as plik:
    linie = plik.readlines()

print(f"Liczba imion: {len(linie)}")
for linia in linie:
    print(linia.strip().upper())
```

> 📝 **Ćwiczenie 6: Dopisywanie do logu**
>
> Za pomocą trybu `"a"` dopisz do pliku `log.txt` (z sekcji 7) nową linię `"Zdarzenie 3"`, nie kasując wcześniejszej zawartości. Na koniec wczytaj i wypisz cały plik, żeby sprawdzić, że wszystko się zgadza.

**👉 Przykładowe rozwiązanie** *(najpierw spróbuj samodzielnie!)*

```python
with open("log.txt", "a", encoding="utf-8") as plik:
    plik.write("Zdarzenie 3\n")

with open("log.txt", "r", encoding="utf-8") as plik:
    print(plik.read())
```

> 📝 **Ćwiczenie 7: Własny wyjątek walidacyjny**
>
> Napisz funkcję `ustaw_haslo(haslo)`, która zgłasza `ValueError` z sensownym komunikatem, jeśli hasło ma mniej niż 8 znaków. W przeciwnym razie zwraca `"Hasło ustawione"`. Przetestuj ją w bloku `try`/`except`.

**👉 Przykładowe rozwiązanie** *(najpierw spróbuj samodzielnie!)*

```python
def ustaw_haslo(haslo):
    if len(haslo) < 8:
        raise ValueError("Hasło musi mieć co najmniej 8 znaków")
    return "Hasło ustawione"

try:
    print(ustaw_haslo("abc"))
except ValueError as blad:
    print(f"Błąd: {blad}")

print(ustaw_haslo("bezpieczne123"))
```

> 🔥 **Ćwiczenie 8 (wyzwanie): Import danych z pliku CSV-podobnego**
>
> Zapisz do pliku `produkty.csv` kilka linii w formacie `nazwa,cena,ilosc` (np. `chleb,4.5,20`), a jedną linię celowo zepsuj (np. brakująca wartość albo cena jako tekst). Wczytaj plik, dla każdej linii rozdziel ją po przecinku (`.split(",")`), spróbuj skonwertować cenę i ilość na liczby w bloku `try`/`except`, pomijając (z wypisanym ostrzeżeniem) linie, których nie da się poprawnie przetworzyć, i wypisz łączną wartość poprawnych produktów.

**👉 Przykładowe rozwiązanie** *(najpierw spróbuj samodzielnie!)*

```python
with open("produkty.csv", "w", encoding="utf-8") as plik:
    plik.write("chleb,4.5,20\n")
    plik.write("mleko,3.2,15\n")
    plik.write("zepsuta_linia,niepoprawna_cena,10\n")
    plik.write("jajka,12.0,8\n")

wartosc_calkowita = 0

with open("produkty.csv", "r", encoding="utf-8") as plik:
    for numer_linii, linia in enumerate(plik, start=1):
        czesci = linia.strip().split(",")
        try:
            nazwa = czesci[0]
            cena = float(czesci[1])
            ilosc = int(czesci[2])
        except (ValueError, IndexError):
            print(f"Pomijam linię {numer_linii} - niepoprawny format: {linia.strip()}")
            continue

        wartosc = cena * ilosc
        print(f"{nazwa}: {wartosc:.2f} zł")
        wartosc_calkowita += wartosc

print(f"Łączna wartość poprawnych produktów: {wartosc_calkowita:.2f} zł")
```

Podpowiedź: `enumerate(plik, start=1)` daje numer linii liczony od 1, co ułatwia wskazanie użytkownikowi, która dokładnie linia w pliku jest zepsuta.

---

### Co dalej?

Gratulacje — to koniec pięciu modułów: od `print()` po pliki i obsługę błędów. To już
solidna podstawa, na której można budować dalej: klasy i programowanie obiektowe,
praca z bibliotekami zewnętrznymi (np. `requests`, `pandas`), albo konkretny projekt,
który połączy to wszystko naraz. Daj znać, w którą stronę chcesz iść dalej.